# Phase 1 LayoutLMv3 字段抽取训练

目标：先用 OCR + Regex 建 baseline，再用 LayoutLMv3 做 SROIE 四字段 token classification。fixture smoke result，不能作为简历指标；真实训练指标未运行时统一标记为 `待真实训练`。

## 1. 环境检查

In [ ]:
import importlib.util
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for package in ['torch', 'transformers', 'PIL']:
    print(package, bool(importlib.util.find_spec(package)))
print('project_root=', PROJECT_ROOT)


## 2. 路径配置

In [ ]:
FIXTURE_RAW = PROJECT_ROOT / 'tests' / 'fixtures' / 'sroie_minimal' / 'raw'
FIXTURE_PROCESSED = PROJECT_ROOT / 'tests' / 'fixtures' / 'sroie_minimal' / 'processed.jsonl'
REAL_TRAIN = PROJECT_ROOT / 'data' / 'phase1' / 'sroie' / 'processed' / 'train.jsonl'
REAL_TEST = PROJECT_ROOT / 'data' / 'phase1' / 'sroie' / 'processed' / 'test.jsonl'
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints' / 'phase1' / 'layoutlmv3'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = REAL_TRAIN if REAL_TRAIN.exists() else FIXTURE_PROCESSED
print('data_path=', DATA_PATH)


## 3. 读取 processed SROIE

In [ ]:
from procureguard.extraction.datasets import iter_sroie_samples, read_processed_jsonl, write_processed_jsonl

if not FIXTURE_PROCESSED.exists():
    write_processed_jsonl(iter_sroie_samples(FIXTURE_RAW), FIXTURE_PROCESSED)
samples = read_processed_jsonl(DATA_PATH)
print('samples=', len(samples))
print('fixture smoke result，不能作为简历指标' if DATA_PATH == FIXTURE_PROCESSED else 'real SROIE data')


## 4. 展示 OCR tokens

In [ ]:
for token in samples[0].tokens[:8]:
    print(token)


## 5. baseline 评测

In [ ]:
import time
from procureguard.extraction.baseline import SroieRegexBaseline
from procureguard.extraction.metrics import build_evaluation_report, metrics_to_markdown

baseline = SroieRegexBaseline()
started_at = time.perf_counter()
baseline_predictions = [baseline.extract(sample.tokens).values() for sample in samples]
references = [sample.labels for sample in samples]
baseline_report = build_evaluation_report(
    baseline_name=baseline.baseline_name,
    predictions=baseline_predictions,
    references=references,
    sample_count=len(samples),
    data_source=str(DATA_PATH),
    is_fixture=DATA_PATH == FIXTURE_PROCESSED,
    started_at=started_at,
)
print(metrics_to_markdown(baseline_report))


## 6. BIO 标签映射

In [ ]:
from procureguard.extraction.alignment import BIO_LABELS, ID2LABEL, LABEL2ID

print(BIO_LABELS)
print(LABEL2ID)
print(ID2LABEL)


## 7. token alignment

In [ ]:
from procureguard.extraction.alignment import align_samples

bio_labels, alignment_summary = align_samples(samples)
print(alignment_summary)
print(list(zip([token.text for token in samples[0].tokens], bio_labels[0])))


## 8. LayoutLMv3Processor 和 Dataset

In [ ]:
from procureguard.extraction.layoutlmv3_dataset import SROIELayoutLMv3Dataset, create_layoutlmv3_processor

# 需要先安装：python -m pip install -e ".[extraction]"
processor = create_layoutlmv3_processor()
dataset = SROIELayoutLMv3Dataset(samples, processor, LABEL2ID, max_length=512)
item = dataset[0]
print(item.keys())


## 9. train / val split 与 DataLoader

In [ ]:
import torch
from torch.utils.data import DataLoader, random_split

val_size = max(1, int(len(dataset) * 0.2)) if len(dataset) > 1 else 1
train_size = max(1, len(dataset) - val_size)
train_dataset, val_dataset = random_split(dataset, [train_size, len(dataset) - train_size])
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1)
batch = next(iter(train_loader))
print(batch.keys())
print({key: tuple(value.shape) for key, value in batch.items()})
print('labels_non_ignore=', int((batch['labels'] != -100).sum().item()))


## 10. 模型、optimizer、scheduler

In [ ]:
from torch.optim import AdamW
from torch.nn.utils import clip_grad_norm_
from transformers import LayoutLMv3ForTokenClassification, get_linear_schedule_with_warmup

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LayoutLMv3ForTokenClassification.from_pretrained(
    'microsoft/layoutlmv3-base',
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
).to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)
num_epochs = 3
total_steps = max(1, len(train_loader) * num_epochs)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)


## 11. 手写训练循环

In [ ]:
def move_batch(batch, device):
    return {key: value.to(device) for key, value in batch.items()}

def train_one_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    total_loss = 0.0
    for batch in dataloader:
        batch = move_batch(batch, device)
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += float(loss.item())
    return total_loss / max(len(dataloader), 1)

@torch.no_grad()
def validate(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    for batch in dataloader:
        batch = move_batch(batch, device)
        outputs = model(**batch)
        total_loss += float(outputs.loss.item())
    return total_loss / max(len(dataloader), 1)


## 12. checkpoint

In [ ]:
best_val_loss = float('inf')
history = []
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss = validate(model, val_loader, device)
    history.append({'epoch': epoch + 1, 'train_loss': train_loss, 'val_loss': val_loss})
    print(history[-1])
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        model.save_pretrained(CHECKPOINT_DIR / 'best')
        processor.save_pretrained(CHECKPOINT_DIR / 'best')

print('best_val_loss=', best_val_loss)


## 13. 字段级 F1、错误分析、对比表

In [ ]:
from procureguard.extraction.error_analysis import collect_error_cases, errors_to_markdown

# TODO: 下一轮把 token logits 解码回字段 JSON，再复用 evaluate_field_f1。
fine_tuned_result = '待真实训练'
errors = collect_error_cases([sample.sample_id for sample in samples], baseline_predictions, references)
print(errors_to_markdown(errors))
print('| method | company_f1 | address_f1 | date_f1 | total_f1 | macro_f1 |')
print('| --- | --- | --- | --- | --- | --- |')
print('| OCR Regex baseline | see report | see report | see report | see report | see report |')
print(f'| LayoutLMv3 fine-tuned | {fine_tuned_result} | {fine_tuned_result} | {fine_tuned_result} | {fine_tuned_result} | {fine_tuned_result} |')


## 14. Phase 1D Task 3 数据与真实 validation

数据源：`Voxel51/scanned_receipts`。本地实际 metadata 为 712 条，按 seed 42 拆成 570 train / 142 validation。评测口径是 `local_validation_split_seed_42`，不是 official test。

In [ ]:
TASK3_ROOT = PROJECT_ROOT / 'data' / 'phase1' / 'sroie_task3'
TASK3_TRAIN = TASK3_ROOT / 'processed' / 'train.jsonl'
TASK3_VALIDATION = TASK3_ROOT / 'processed' / 'validation.jsonl'
TASK3_BASELINE_REPORT = PROJECT_ROOT / 'reports' / 'phase1' / 'baseline_sroie_task3_validation.json'
TASK3_ALIGNMENT_REPORT = PROJECT_ROOT / 'reports' / 'phase1' / 'sroie_task3_alignment_summary.md'

task3_train = read_processed_jsonl(TASK3_TRAIN)
task3_validation = read_processed_jsonl(TASK3_VALIDATION)
print('train=', len(task3_train), 'validation=', len(task3_validation))
print('evaluation_split=local_validation_split_seed_42')


## 15. 真实 baseline 与 alignment 报告

当前真实 validation OCR + Regex macro F1 为 0.4387。LayoutLMv3 F1 仍为 `待真实训练`。

In [ ]:
import json

task3_report = json.loads(TASK3_BASELINE_REPORT.read_text(encoding='utf-8'))
print(metrics_to_markdown(task3_report))
print(TASK3_ALIGNMENT_REPORT.read_text(encoding='utf-8'))


## 16. 有效 Dataset / forward 与 PaddleOCR smoke

- Task 3 batch smoke：labels_non_o_count = 6
- Task 3 单 batch forward：CPU，loss = 2.216274，logits shape = (1, 512, 9)
- PaddleOCR 3.6.0 + PaddlePaddle 3.3.1：3 张 ModelScope 真实图片端到端 smoke 已完成
- 下一步：在 Colab 或 ModelScope GPU Notebook 中运行完整 fine-tuning，记录真实 validation loss 与字段级 F1